In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
# load data
df = pd.read_csv("../data/processed/final_data.csv")

print(df.shape)
print(df.head())
print(df.columns.tolist())

(7043, 30)
   customerID  gender  SeniorCitizen  Dependents  tenure  PhoneService  \
0  1024-GUALD  Female              0           0       1             0   
1  0484-JPBRU    Male              0           0      41             1   
2  3620-EHIMZ  Female              0           1      52             1   
3  6910-HADCM  Female              0           0       1             1   
4  8587-XYZSF    Male              0           0      67             1   

   DeviceProtection  TechSupport        Contract  PaperlessBilling  ...  \
0                 0            0  Month-to-month                 1  ...   
1                 0            0  Month-to-month                 1  ...   
2                 0            0        Two year                 0  ...   
3                 1            0  Month-to-month                 0  ...   
4                 0            1        Two year                 0  ...   

  high_value_customer  has_complaint  has_internet  fiber_user  \
0                   0      

In [3]:
# remove rows where target is missing
df = df.dropna(subset=["Churn"]).copy()

In [4]:
# target
y = df["Churn"]

# dropping columns that should not go into structured model
drop_cols = ["customerID", "Churn", "complaints"]

X = df.drop(columns=drop_cols, errors="ignore")

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 27)
y shape: (7043,)


In [5]:
# identifying numeric and categorical columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

Numeric columns:
['SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'DeviceProtection', 'TechSupport', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'num_complaints', 'payment_risk', 'low_engagement', 'support_calls', 'high_value_customer', 'has_complaint', 'has_internet', 'fiber_user', 'complaint_text_length', 'complaint_negative_score', 'complaint_negative_flag', 'billing_issue_flag', 'technical_issue_flag', 'service_issue_flag']

Categorical columns:
['gender', 'Contract', 'PaymentMethod', 'tenure_group']


In [6]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5634, 27)
Test shape: (1409, 27)


In [7]:
# preprocessing for numeric columns
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# preprocessing for categorical columns
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [8]:
# logistic regression pipeline
log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# random forest pipeline
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42
    ))
])

In [9]:
# fitting models
log_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen',
                                                   'Dependents', 'tenure',
                                                   'PhoneService',
                                                   'DeviceProtection',
                                                   'TechSupport',
                                                   'PaperlessBilling',
                                                   'MonthlyCharges',
                                                   'TotalCharges',
                                                   'num_complaints',
                                                   'payment_risk',
                                                   'low_engagement',
                                                   '...
                                                   'technical_issue_flag',
                                                   'service_issue_flag']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'Contract',
                                                   'PaymentMethod',
                                                   'tenure_group'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=8, min_samples_leaf=4,
                                        min_samples_split=10, n_estimators=200,
                                        random_state=42))])

In [10]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n{'=' * 50}")
    print(model_name)
    print(f"{'=' * 50}")
    print("Accuracy  :", round(acc, 4))
    print("Precision :", round(prec, 4))
    print("Recall    :", round(rec, 4))
    print("F1 Score  :", round(f1, 4))
    print("ROC AUC   :", round(roc, 4))
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return {
        "model": model_name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": roc
    }

In [11]:
log_results = evaluate_model(log_model, X_test, y_test, "Logistic Regression")



Logistic Regression
Accuracy  : 0.7913
Precision : 0.6342
Recall    : 0.5053
F1 Score  : 0.5625
ROC AUC   : 0.8318

Confusion Matrix:
[[926 109]
 [185 189]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [12]:
rf_results = evaluate_model(rf_model, X_test, y_test, "Random Forest")


Random Forest
Accuracy  : 0.7871
Precision : 0.6267
Recall    : 0.4893
F1 Score  : 0.5495
ROC AUC   : 0.835

Confusion Matrix:
[[926 109]
 [191 183]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [13]:
results_df = pd.DataFrame([log_results, rf_results])
results_df = results_df.sort_values(by="f1", ascending=False)

print(results_df)

                 model  accuracy  precision    recall       f1   roc_auc
0  Logistic Regression  0.791341   0.634228  0.505348  0.56250  0.831827
1        Random Forest  0.787083   0.626712  0.489305  0.54955  0.834974


In [14]:
best_model = rf_model
best_model_name = "random_forest"

os.makedirs("models", exist_ok=True)

joblib.dump(best_model, f"models/{best_model_name}_pipeline.pkl")

print(f"Saved model: models/{best_model_name}_pipeline.pkl")

Saved model: models/random_forest_pipeline.pkl


In [15]:
preprocessor.fit(X_train)

joblib.dump(preprocessor, "../models/preprocessor.pkl")

print("Saved preprocessor: models/preprocessor.pkl")

Saved preprocessor: models/preprocessor.pkl


In [16]:
sample_row = X_test.iloc[[0]]

sample_pred = best_model.predict(sample_row)[0]
sample_prob = best_model.predict_proba(sample_row)[0][1]

print("Predicted churn:", sample_pred)
print("Predicted churn probability:", round(sample_prob, 4))
print("\nSample input row:")
print(sample_row)

Predicted churn: 1
Predicted churn probability: 0.7382

Sample input row:
     gender  SeniorCitizen  Dependents  tenure  PhoneService  \
456  Female              1           0       2             1   

     DeviceProtection  TechSupport        Contract  PaperlessBilling  \
456                 0            0  Month-to-month                 1   

        PaymentMethod  ...  high_value_customer  has_complaint  has_internet  \
456  Electronic check  ...                    1              1             1   

     fiber_user  complaint_text_length  complaint_negative_score  \
456           1                     93                         0   

    complaint_negative_flag  billing_issue_flag  technical_issue_flag  \
456                       0                   0                     0   

     service_issue_flag  
456                   0  

[1 rows x 27 columns]


In [17]:
test_results = X_test.copy()
test_results["actual_churn"] = y_test.values
test_results["predicted_churn"] = best_model.predict(X_test)
test_results["churn_probability"] = best_model.predict_proba(X_test)[:, 1]

test_results.to_csv("../data/processed/test_predictions.csv", index=False)
print("Saved test predictions")

Saved test predictions


In [18]:
test_results.sort_values("churn_probability", ascending=False).head(10)

,gender,SeniorCitizen,Dependents,tenure,PhoneService,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,...,fiber_user,complaint_text_length,complaint_negative_score,complaint_negative_flag,billing_issue_flag,technical_issue_flag,service_issue_flag,actual_churn,predicted_churn,churn_probability
4942,Female,1,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,64,0,0,0,0,0,1,1,0.865210
6026,Male,1,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,93,0,0,0,1,0,1,1,0.861450
6770,Male,0,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,259,0,0,0,1,0,1,1,0.850559
680,Female,0,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,100,1,1,0,0,0,1,1,0.850210
835,Female,1,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,0,0,0,0,0,0,1,1,0.849981
2114,Male,0,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,47,0,0,0,0,0,0,1,0.849810
726,Male,0,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,205,0,0,0,0,0,1,1,0.847539
1081,Female,1,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,0,0,0,0,0,0,1,1,0.843753
3654,Male,0,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,0,0,0,0,0,0,1,1,0.843507
1386,Female,1,0,1,1,0,0,Month-to-month,1,Electronic check,...,1,518,0,0,0,0,0,1,1,0.843283


In [19]:
test_results.sort_values("churn_probability", ascending=True).head(10)

,gender,SeniorCitizen,Dependents,tenure,PhoneService,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,...,fiber_user,complaint_text_length,complaint_negative_score,complaint_negative_flag,billing_issue_flag,technical_issue_flag,service_issue_flag,actual_churn,predicted_churn,churn_probability
6798,Female,0,0,68,1,0,0,Two year,0,Credit card (automatic),...,0,132,0,0,0,1,0,0,0,0.001178
3573,Female,0,1,71,1,0,0,Two year,0,Bank transfer (automatic),...,0,341,0,0,0,1,0,0,0,0.001231
1785,Female,0,0,68,1,0,0,Two year,0,Bank transfer (automatic),...,0,129,0,0,0,0,0,0,0,0.001398
94,Female,0,0,72,1,0,0,Two year,0,Credit card (automatic),...,0,78,0,0,0,0,0,0,0,0.001645
274,Female,0,0,71,1,0,0,Two year,0,Bank transfer (automatic),...,0,164,0,0,0,1,0,0,0,0.001647
6687,Female,0,1,71,1,0,0,Two year,0,Bank transfer (automatic),...,0,99,0,0,0,0,0,0,0,0.001691
3482,Female,0,0,68,1,0,0,Two year,0,Bank transfer (automatic),...,0,128,0,0,0,0,0,0,0,0.001741
1900,Female,0,0,62,1,0,0,Two year,0,Bank transfer (automatic),...,0,180,0,0,1,1,0,0,0,0.001753
1057,Male,0,1,72,1,0,0,Two year,0,Credit card (automatic),...,0,85,0,0,0,0,0,0,0,0.001927
2361,Female,0,0,72,1,0,0,Two year,0,Credit card (automatic),...,0,116,0,0,0,0,1,0,0,0.002083
